importing NLTK and downloading browncorpus data.

In [11]:
import nltk
nltk.download('brown')
nltk.download('universal_tagset')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


True

preprocessing

In [12]:
from collections import defaultdict
import numpy as np

# Hardcoded tagged sentences (word, POS tag)
sentences = [
    [("The", "DET"), ("cat", "NOUN"), ("sat", "VERB"), ("on", "ADP"), ("the", "DET"), ("mat", "NOUN")],
    [("She", "PRON"), ("is", "VERB"), ("reading", "VERB"), ("a", "DET"), ("book", "NOUN")],
    [("Dogs", "NOUN"), ("are", "VERB"), ("running", "VERB"), ("in", "ADP"), ("park", "NOUN")],
    [("Hello", "INTJ"), ("world", "NOUN")],
    [("AI", "NOUN"), ("is", "VERB"), ("fun", "ADJ")]
]

processed_sentences = []

def clean_word(word):
    word = word.lower()
    if word.isalpha():
        return word
    return None

# Same processing logic
for sent in sentences:
    cleaned = []
    for word, tag in sent:
        w = clean_word(word)
        if w:
            cleaned.append((w, tag))
    if len(cleaned) > 1:
        processed_sentences.append(cleaned)

# Print result to verify
for sent in processed_sentences:
    print(sent)

[('the', 'DET'), ('cat', 'NOUN'), ('sat', 'VERB'), ('on', 'ADP'), ('the', 'DET'), ('mat', 'NOUN')]
[('she', 'PRON'), ('is', 'VERB'), ('reading', 'VERB'), ('a', 'DET'), ('book', 'NOUN')]
[('dogs', 'NOUN'), ('are', 'VERB'), ('running', 'VERB'), ('in', 'ADP'), ('park', 'NOUN')]
[('hello', 'INTJ'), ('world', 'NOUN')]
[('ai', 'NOUN'), ('is', 'VERB'), ('fun', 'ADJ')]


HMM components

In [13]:
states = set()
vocab = set()

for sent in processed_sentences:
    for word, tag in sent:
        states.add(tag)
        vocab.add(word)

states = list(states)
vocab = list(vocab)

print("Number of states:", len(states))
print("Vocabulary size:", len(vocab))

Number of states: 7
Vocabulary size: 19


transition count

In [14]:
transition_counts = defaultdict(lambda: defaultdict(int))
tag_counts = defaultdict(int)

for sent in processed_sentences:
    for i in range(len(sent) - 1):
        tag1 = sent[i][1]
        tag2 = sent[i+1][1]
        transition_counts[tag1][tag2] += 1
        tag_counts[tag1] += 1

Emission counts

In [15]:
emission_counts = defaultdict(lambda: defaultdict(int))

for sent in processed_sentences:
    for word, tag in sent:
        emission_counts[tag][word] += 1


counts to probability smoothing

In [16]:
alpha = 1  # Laplace smoothing

transition_prob = defaultdict(dict)
for tag1 in transition_counts:
    total = sum(transition_counts[tag1].values())
    for tag2 in transition_counts[tag1]:
        transition_prob[tag1][tag2] = (transition_counts[tag1][tag2] + alpha) / (total + alpha * len(states))


emission_prob = defaultdict(dict)
for tag in emission_counts:
    total = sum(emission_counts[tag].values())
    for word in vocab:
        emission_prob[tag][word] = (emission_counts[tag][word] + alpha) / (total + alpha * len(vocab))

word to tag index

In [17]:
word_to_tags = defaultdict(set)

for tag in emission_counts:
    for word in emission_counts[tag]:
        word_to_tags[word].add(tag)


word prediction

In [18]:
def predict_next_word(current_word):
    current_word = current_word.lower()

    # unknown word fallback
    if current_word not in word_to_tags:
        return "the"

    best_word = None
    best_score = 0

    for tag in word_to_tags[current_word]:

        if tag not in transition_prob:
            continue

        for next_tag in transition_prob[tag]:
            trans_p = transition_prob[tag][next_tag]

            # take only top 20 most probable words for that tag
            sorted_words = sorted(
                emission_prob[next_tag].items(),
                key=lambda x: x[1],
                reverse=True
            )[:20]

            for word, emit_p in sorted_words:
                score = trans_p * emit_p

                if score > best_score:
                    best_score = score
                    best_word = word

    return best_word


running the model

In [20]:
while True:
    w = input("Enter word: ")
    if w == "exit;":
        break
    print("Predicted next word:", predict_next_word(w))


Enter word: the
Predicted next word: is
Enter word: man
Predicted next word: the
Enter word: exit;
